# LeetCode 1800 — Maximum Ascending Subarray Sum bằng Python

Link đề: https://leetcode.com/problems/maximum-ascending-subarray-sum/

Độ khó: **Easy** · Chủ đề: Array

```python
class Solution:
    def maxAscendingSum(self, nums: List[int]) -> int:

```

**Mục tiêu học:**

1. Phân biệt **subarray** (mảng con liên tiếp) với **subsequence** (dãy con không cần liên tiếp).
2. Hiểu "**tăng ngặt** (strictly increasing)": `a < b < c`, không cho bằng nhau.
3. Viết baseline duyệt mọi subarray `O(n²)` / `O(n³)` và thấy nó làm thừa.
4. Học kỹ thuật **tổng chạy + reset** trong một lượt `O(n)` (họ hàng của thuật toán Kadane).
5. Biết chứng minh: các đoạn tăng tối đa **không chồng nhau**, mỗi đoạn kết thúc ở chỗ "gãy".

## 1. Bài toán yêu cầu gì?

Cho mảng **số nguyên dương** `nums`. Trả về **tổng lớn nhất** của một subarray **tăng ngặt**.

- **Subarray** = dãy các phần tử **liên tiếp** của mảng.
- Một phần tử đơn lẻ cũng là subarray tăng ngặt (độ dài 1).

| `nums` | Subarray tốt nhất | Tổng |
|---|---|:---:|
| `[10,20,30,5,10,50]` | `[5,10,50]` | `65` |
| `[10,20,30,40,50]` | toàn bộ | `150` |
| `[12,17,15,13,10,11,12]` | `[10,11,12]` | `33` |
| `[100,10,1]` | `[100]` | `100` |
| `[3,3,3]` | `[3]` (bằng nhau không tính là tăng) | `3` |

**Ràng buộc:** `1 <= len(nums) <= 100`, `1 <= nums[i] <= 100`.

In [1]:
from typing import List

examples = [
    ([10, 20, 30, 5, 10, 50], 65),
    ([10, 20, 30, 40, 50], 150),
    ([12, 17, 15, 13, 10, 11, 12], 33),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[10, 20, 30, 5, 10, 50] -> kỳ vọng 65
[10, 20, 30, 40, 50] -> kỳ vọng 150
[12, 17, 15, 13, 10, 11, 12] -> kỳ vọng 33


## 2. Nền tảng Python và khái niệm

### 2.1. Slicing `nums[i:j]` — lấy subarray

`nums[i:j]` lấy các phần tử từ chỉ số `i` tới `j - 1` (không gồm `j`). Đây chính là một subarray.

In [2]:
nums = [10, 20, 30, 5, 10, 50]
print("nums[3:6] =", nums[3:6])
print("nums[0:3] =", nums[0:3])
print("nums[2:3] =", nums[2:3], "(1 phần tử)")
print("sum(nums[3:6]) =", sum(nums[3:6]))

nums[3:6] = [5, 10, 50]
nums[0:3] = [10, 20, 30]
nums[2:3] = [30] (1 phần tử)
sum(nums[3:6]) = 65


### 2.2. Subarray vs subsequence

`[10, 20, 50]` là **dãy con tăng** (bỏ bớt phần tử ở giữa) nhưng **không phải** subarray vì `10, 20` và `50` không liền nhau. Bài này chỉ chấp nhận subarray.

### 2.3. Kiểm tra một list có tăng ngặt không

Dùng `all(...)` — trả `True` nếu **mọi** điều kiện bên trong đều đúng. `zip(a, a[1:])` ghép từng cặp liền kề `(a[k], a[k+1])`.

In [3]:
def is_strictly_increasing(arr):
    return all(x < y for x, y in zip(arr, arr[1:]))

print(list(zip([5, 10, 50], [10, 50])))
print(is_strictly_increasing([5, 10, 50]))   # True
print(is_strictly_increasing([5, 5, 50]))    # False (bằng nhau)
print(is_strictly_increasing([7]))           # True (1 phần tử)

[(5, 10), (10, 50)]
True
False
True


### 2.4. `max` với nhiều đối số

`max(a, b)` trả giá trị lớn hơn — dùng để cập nhật "kỷ lục" `best = max(best, current)`.

In [4]:
best = 60
current = 65
best = max(best, current)
print(best)

65


## 3. Cách trực giác — thử mọi subarray (brute force)

Thử mọi cặp `(i, j)`, kiểm tra `nums[i:j+1]` có tăng ngặt không, nếu có thì so tổng với kỷ lục.

In [5]:
def max_ascending_sum_brute(nums: List[int]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        for j in range(i, n):
            sub = nums[i:j + 1]
            if is_strictly_increasing(sub):
                best = max(best, sum(sub))
    return best

for nums, expected in examples:
    print(max_ascending_sum_brute(nums), "| kỳ vọng", expected)

65 | kỳ vọng 65
150 | kỳ vọng 150
33 | kỳ vọng 33


### 3.1. Chi phí và cải tiến nhỏ

- `O(n²)` cặp `(i, j)` × `O(n)` để cắt, kiểm tra, cộng → `O(n³)`.
- Cải tiến: cố định `i`, **kéo dài** `j` sang phải chừng nào còn tăng, cộng dồn tổng; gặp chỗ không tăng thì dừng (mọi `j` xa hơn cũng không tăng). → `O(n²)`.

In [6]:
def max_ascending_sum_extend(nums: List[int]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        total = nums[i]
        best = max(best, total)
        j = i + 1
        while j < n and nums[j] > nums[j - 1]:
            total += nums[j]
            best = max(best, total)
            j += 1
    return best

for nums, expected in examples:
    print(max_ascending_sum_extend(nums), "| kỳ vọng", expected)

65 | kỳ vọng 65
150 | kỳ vọng 150
33 | kỳ vọng 33


## 4. Nút thắt và insight

Cách "kéo dài" vẫn lặp lại việc: bắt đầu từ `i = 0` ta đã cộng `10+20+30`; bắt đầu từ `i = 1` lại cộng `20+30`… Những subarray bắt đầu giữa một đoạn tăng **luôn có tổng nhỏ hơn** subarray bắt đầu từ đầu đoạn (vì mọi số đều **dương**, thêm phần tử đầu chỉ làm tổng lớn lên).

> **Insight 1:** Mảng tự chia thành các **đoạn tăng tối đa** (maximal runs), cắt tại mỗi chỗ `nums[k] <= nums[k-1]`. Đáp án = tổng lớn nhất trong các đoạn đó.

> **Insight 2:** Ta có thể tính tổng mỗi đoạn trong **một lượt**: giữ `current` = tổng đoạn tăng **kết thúc tại vị trí hiện tại**. Nếu `nums[k] > nums[k-1]` thì `current += nums[k]`, ngược lại **reset** `current = nums[k]`.

(Gợi ý của đề cũng nói: *"điểm kết thúc của một đoạn tăng là điểm bắt đầu của đoạn kế tiếp"*.)

In [7]:
# Nhìn các đoạn tăng tối đa bằng mắt
nums = [10, 20, 30, 5, 10, 50]
runs = [[nums[0]]]
for k in range(1, len(nums)):
    if nums[k] > nums[k - 1]:
        runs[-1].append(nums[k])
    else:
        runs.append([nums[k]])
print("các đoạn tăng tối đa:", runs)
print("tổng từng đoạn:", [sum(r) for r in runs])

các đoạn tăng tối đa: [[10, 20, 30], [5, 10, 50]]
tổng từng đoạn: [60, 65]


`runs[-1].append(...)` thêm vào **đoạn cuối cùng** đang xây; `runs.append([x])` mở một đoạn mới chỉ có `x`.

## 5. Lời giải chính — tổng chạy + reset

**Bước 1.** `current = best = nums[0]` (đoạn đầu chỉ có phần tử đầu).

**Bước 2.** Với `k = 1 .. n-1`:
- Nếu `nums[k] > nums[k-1]`: nối tiếp đoạn → `current += nums[k]`.
- Ngược lại: đoạn bị gãy → `current = nums[k]` (bắt đầu đoạn mới).
- `best = max(best, current)`.

**Bước 3.** Trả `best`.

In [8]:
def max_ascending_sum(nums: List[int]) -> int:
    current = best = nums[0]
    for k in range(1, len(nums)):
        if nums[k] > nums[k - 1]:
            current += nums[k]
        else:
            current = nums[k]
        best = max(best, current)
    return best

for nums, expected in examples:
    print(max_ascending_sum(nums), "| kỳ vọng", expected)

65 | kỳ vọng 65
150 | kỳ vọng 150
33 | kỳ vọng 33


Tại sao bắt đầu từ `k = 1`? Vì ta so `nums[k]` với `nums[k-1]`; với `k = 0` thì `nums[-1]` là phần tử **cuối mảng** — không có nghĩa "phần tử trước". Phần tử đầu đã được xử lý ở bước khởi tạo.

## 6. TRACE từng bước

Trace `nums = [12, 17, 15, 13, 10, 11, 12]`:

| k | nums[k-1] | nums[k] | tăng? | current | best |
|---|---|---|---|---|---|
| 0 | – | 12 | (khởi tạo) | 12 | 12 |
| 1 | 12 | 17 | có | 29 | 29 |
| 2 | 17 | 15 | không → reset | 15 | 29 |
| 3 | 15 | 13 | không → reset | 13 | 29 |
| 4 | 13 | 10 | không → reset | 10 | 29 |
| 5 | 10 | 11 | có | 21 | 29 |
| 6 | 11 | 12 | có | 33 | **33** |

In [9]:
def trace_max_ascending_sum(nums: List[int]) -> int:
    print("nums =", nums)
    current = best = nums[0]
    print(f"k=0: khởi tạo current={current}, best={best}")
    for k in range(1, len(nums)):
        if nums[k] > nums[k - 1]:
            current += nums[k]
            action = f"{nums[k-1]} < {nums[k]}: nối tiếp"
        else:
            current = nums[k]
            action = f"{nums[k-1]} >= {nums[k]}: RESET"
        best = max(best, current)
        print(f"k={k}: {action:22} | current={current:3}, best={best:3}")
    print("-> kết quả", best)
    return best

trace_max_ascending_sum([12, 17, 15, 13, 10, 11, 12])

nums = [12, 17, 15, 13, 10, 11, 12]
k=0: khởi tạo current=12, best=12
k=1: 12 < 17: nối tiếp      | current= 29, best= 29
k=2: 17 >= 15: RESET        | current= 15, best= 29
k=3: 15 >= 13: RESET        | current= 13, best= 29
k=4: 13 >= 10: RESET        | current= 10, best= 29
k=5: 10 < 11: nối tiếp      | current= 21, best= 29
k=6: 11 < 12: nối tiếp      | current= 33, best= 33
-> kết quả 33


33

### 6.1. Trace trường hợp có phần tử bằng nhau và phần tử đầu là đáp án

In [10]:
trace_max_ascending_sum([3, 3, 3])
print()
trace_max_ascending_sum([100, 10, 1])
print()
trace_max_ascending_sum([10, 20, 30, 5, 10, 50])

nums = [3, 3, 3]
k=0: khởi tạo current=3, best=3
k=1: 3 >= 3: RESET          | current=  3, best=  3
k=2: 3 >= 3: RESET          | current=  3, best=  3
-> kết quả 3

nums = [100, 10, 1]
k=0: khởi tạo current=100, best=100
k=1: 100 >= 10: RESET       | current= 10, best=100
k=2: 10 >= 1: RESET         | current=  1, best=100
-> kết quả 100

nums = [10, 20, 30, 5, 10, 50]
k=0: khởi tạo current=10, best=10
k=1: 10 < 20: nối tiếp      | current= 30, best= 30
k=2: 20 < 30: nối tiếp      | current= 60, best= 60
k=3: 30 >= 5: RESET         | current=  5, best= 60
k=4: 5 < 10: nối tiếp       | current= 15, best= 60
k=5: 10 < 50: nối tiếp      | current= 65, best= 65
-> kết quả 65


65

Với `[3, 3, 3]`: `3 >= 3` nên **reset** — vì "tăng ngặt" không chấp nhận bằng nhau. Nếu viết nhầm `>=` thì sẽ ra 9.

## 7. Các lời giải khác

### Cách 4 — Tách đoạn rồi lấy max tổng

Dùng đúng đoạn code ở mục 4, gói thành hàm. Dễ hiểu, nhưng tốn bộ nhớ `O(n)` để giữ các đoạn.

In [11]:
def max_ascending_sum_runs(nums: List[int]) -> int:
    runs = [[nums[0]]]
    for k in range(1, len(nums)):
        if nums[k] > nums[k - 1]:
            runs[-1].append(nums[k])
        else:
            runs.append([nums[k]])
    return max(sum(r) for r in runs)

for nums, expected in examples:
    print(max_ascending_sum_runs(nums), "| kỳ vọng", expected)

65 | kỳ vọng 65
150 | kỳ vọng 150
33 | kỳ vọng 33


### Cách 5 — Dùng `zip` duyệt cặp liền kề

Thay vì chỉ số, ghép `(prev, cur)` bằng `zip(nums, nums[1:])` — code đọc như văn.

In [12]:
def max_ascending_sum_zip(nums: List[int]) -> int:
    current = best = nums[0]
    for prev, cur in zip(nums, nums[1:]):
        current = current + cur if cur > prev else cur
        best = max(best, current)
    return best

for nums, expected in examples:
    print(max_ascending_sum_zip(nums), "| kỳ vọng", expected)

65 | kỳ vọng 65
150 | kỳ vọng 150
33 | kỳ vọng 33


### 7.1. Bảng so sánh

| Cách | Ý tưởng | Time | Space |
|---|---|---|---|
| 1. Brute force | mọi subarray, kiểm tra tăng | `O(n³)` | `O(n)` (slice) |
| 2. Kéo dài từ mỗi `i` | dừng khi gãy | `O(n²)` | `O(1)` |
| **3. Tổng chạy + reset (chính)** | một lượt | `O(n)` | `O(1)` |
| 4. Tách đoạn | max tổng các đoạn | `O(n)` | `O(n)` |
| 5. `zip` cặp liền kề | như cách 3 | `O(n)` | `O(n)` (do `nums[1:]` tạo bản sao) |

## 8. Code nộp LeetCode

In [13]:
class Solution:
    def maxAscendingSum(self, nums: List[int]) -> int:
        current = best = nums[0]
        for k in range(1, len(nums)):
            if nums[k] > nums[k - 1]:
                current += nums[k]
            else:
                current = nums[k]
            best = max(best, current)
        return best

sol = Solution()
for nums, expected in examples:
    print(sol.maxAscendingSum(nums), "| kỳ vọng", expected)

65 | kỳ vọng 65
150 | kỳ vọng 150
33 | kỳ vọng 33


## 9. Vì sao thuật toán đúng?

**Invariant:** sau khi xử lý chỉ số `k`,

- `current` = tổng của subarray tăng ngặt **dài nhất kết thúc tại `k`** (bắt đầu từ đầu đoạn tăng chứa `k`);
- `best` = tổng lớn nhất của mọi subarray tăng ngặt nằm trong `nums[0..k]`.

Giữ invariant: nếu `nums[k] > nums[k-1]`, đoạn tăng dài nhất kết thúc tại `k` = đoạn tại `k-1` nối thêm `nums[k]` → cộng. Nếu không, không subarray tăng nào chứa cả `k-1` và `k` → đoạn dài nhất kết thúc tại `k` chỉ là `[nums[k]]` → reset.

Vì sao chỉ cần đoạn **dài nhất** kết thúc tại `k`? Mọi subarray tăng kết thúc tại `k` là **hậu tố** của đoạn đó; vì các số đều **dương**, đoạn dài nhất có tổng lớn nhất. Mọi subarray tăng đều kết thúc tại một `k` nào đó → `best` xét đủ.

(Chú ý: lập luận "dài hơn ⇒ tổng lớn hơn" dựa vào `nums[i] >= 1`. Nếu đề cho phép số âm thì lập luận này **không còn đúng** — phần đầu âm của đoạn tăng có thể nên bỏ đi; xem biến thể 14.2.)

## 10. Độ phức tạp

`n = len(nums)`.

- **Time:** `O(n)` — một vòng lặp.
- **Space:** `O(1)` — hai biến `current`, `best`.

## 11. Bộ test

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual}")

sol = Solution()
check("đề 1", sol.maxAscendingSum([10, 20, 30, 5, 10, 50]), 65)
check("đề 2", sol.maxAscendingSum([10, 20, 30, 40, 50]), 150)
check("đề 3", sol.maxAscendingSum([12, 17, 15, 13, 10, 11, 12]), 33)
check("1 phần tử", sol.maxAscendingSum([7]), 7)
check("toàn bằng nhau", sol.maxAscendingSum([3, 3, 3]), 3)
check("giảm dần", sol.maxAscendingSum([100, 10, 1]), 100)
check("đoạn tốt ở đầu", sol.maxAscendingSum([1, 50, 2, 3]), 51)
check("bằng nhau cắt đoạn", sol.maxAscendingSum([1, 2, 2, 3]), 5)
check("phần tử đơn lớn hơn đoạn", sol.maxAscendingSum([1, 2, 3, 100, 1]), 106)

PASS đề 1: 65
PASS đề 2: 150
PASS đề 3: 33
PASS 1 phần tử: 7
PASS toàn bằng nhau: 3
PASS giảm dần: 100
PASS đoạn tốt ở đầu: 51
PASS bằng nhau cắt đoạn: 5
PASS phần tử đơn lớn hơn đoạn: 106


In [15]:
import random
random.seed(1800)
solvers = [max_ascending_sum_brute, max_ascending_sum_extend, max_ascending_sum,
           max_ascending_sum_runs, max_ascending_sum_zip, Solution().maxAscendingSum]
for _ in range(2000):
    n = random.randint(1, 12)
    nums = [random.randint(1, 10) for _ in range(n)]   # giá trị nhỏ -> nhiều số bằng nhau
    answers = [f(nums) for f in solvers]
    assert len(set(answers)) == 1, (nums, answers)
print("6 cách khớp nhau trên 2000 mảng ngẫu nhiên (seed 1800)")

6 cách khớp nhau trên 2000 mảng ngẫu nhiên (seed 1800)


Chọn giá trị trong `1..10` (thay vì `1..100`) để **cố tình** tạo nhiều phần tử bằng nhau — chỗ dễ sai nhất.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Dùng `>=` (cho phép bằng nhau)

In [16]:
def wrong_ge(nums):
    current = best = nums[0]
    for k in range(1, len(nums)):
        if nums[k] >= nums[k - 1]:
            current += nums[k]
        else:
            current = nums[k]
        best = max(best, current)
    return best

print("sai:", wrong_ge([3, 3, 3]), "| đúng:", max_ascending_sum([3, 3, 3]))

sai: 9 | đúng: 3


### Lỗi 2: Cập nhật `best` chỉ khi reset

Nếu chỉ viết `best = max(best, current)` **trong nhánh else** (trước khi reset), đoạn tăng **cuối cùng** không bao giờ được so sánh.

In [17]:
def wrong_update_only_on_break(nums):
    current = best = nums[0]
    for k in range(1, len(nums)):
        if nums[k] > nums[k - 1]:
            current += nums[k]
        else:
            best = max(best, current)
            current = nums[k]
    return best

print("sai:", wrong_update_only_on_break([10, 20, 30, 5, 10, 50]), "| đúng:", max_ascending_sum([10, 20, 30, 5, 10, 50]))

sai: 60 | đúng: 65


**Sửa:** cập nhật `best` sau **mỗi** bước, hoặc thêm `return max(best, current)` sau vòng lặp.

### Lỗi 3: Reset `current = 0` thay vì `current = nums[k]`

Quên rằng phần tử gây gãy chính là **phần tử đầu** của đoạn mới.

In [18]:
def wrong_reset_zero(nums):
    current = best = nums[0]
    for k in range(1, len(nums)):
        if nums[k] > nums[k - 1]:
            current += nums[k]
        else:
            current = 0
        best = max(best, current)
    return best

print("sai:", wrong_reset_zero([10, 20, 30, 5, 10, 50]), "| đúng:", max_ascending_sum([10, 20, 30, 5, 10, 50]))

sai: 60 | đúng: 65


### Lỗi 4: Nhầm subarray với subsequence

Lấy `[10, 20, 30, 50]` (bỏ qua `5, 10`) cho tổng 110 — không hợp lệ vì không liên tiếp.

### Lỗi 5: Khởi tạo `best = 0` rồi bắt đầu vòng lặp từ `k = 1`

Nếu `best = 0` và không tính phần tử đầu, mảng `[100, 10, 1]` sẽ cho 10 thay vì 100. Khởi tạo `current = best = nums[0]` tránh được lỗi này.

In [19]:
def wrong_init(nums):
    current = nums[0]
    best = 0
    for k in range(1, len(nums)):
        current = current + nums[k] if nums[k] > nums[k - 1] else nums[k]
        best = max(best, current)
    return best

print("sai:", wrong_init([100, 10, 1]), "| đúng:", max_ascending_sum([100, 10, 1]))

sai: 10 | đúng: 100


## 13. Mẹo nhận dạng pattern

- "**Subarray liên tiếp** + điều kiện giữa hai phần tử liền kề (tăng, bằng, xen kẽ…)" → **một lượt, giữ trạng thái của đoạn kết thúc tại vị trí hiện tại, reset khi gãy**.
- "Tổng/độ dài **lớn nhất** của đoạn thỏa điều kiện" → `best = max(best, current)` sau mỗi bước.
- Đây là khung của Kadane (53 Maximum Subarray), 674 Longest Continuous Increasing Subsequence, 485 Max Consecutive Ones, 1446 Consecutive Characters.

## 14. Biến thể

### 14.1. Độ dài đoạn tăng ngặt dài nhất (LeetCode 674)

Đổi "tổng" thành "độ dài": `current += 1` thay vì `+= nums[k]`, reset về `1`.

In [20]:
def longest_increasing_run(nums: List[int]) -> int:
    current = best = 1
    for k in range(1, len(nums)):
        current = current + 1 if nums[k] > nums[k - 1] else 1
        best = max(best, current)
    return best

print(longest_increasing_run([1, 3, 5, 4, 7]), longest_increasing_run([2, 2, 2]))   # 3 1

3 1


### 14.2. Cho phép số âm

Với số âm, đoạn tăng dài nhất kết thúc tại `k` **không** chắc có tổng lớn nhất (ví dụ `[-5, 1, 2]`: tốt nhất là `[1, 2]` = 3, không phải cả đoạn = -2). Kết hợp với Kadane: khi nối tiếp, chọn `max(current + x, x)`.

In [21]:
def max_ascending_sum_with_negatives(nums: List[int]) -> int:
    current = best = nums[0]
    for k in range(1, len(nums)):
        if nums[k] > nums[k - 1]:
            current = max(current + nums[k], nums[k])   # có thể bỏ phần đầu âm
        else:
            current = nums[k]
        best = max(best, current)
    return best

print(max_ascending_sum_with_negatives([-5, 1, 2]))        # 3
print(max_ascending_sum([-5, 1, 2]), "<- cách gốc sai khi có số âm")

3
-2 <- cách gốc sai khi có số âm


## 15. Bài tập luyện tập

**Bài 1 (tính tay):** Chia `[5, 6, 1, 2, 3, 2, 9]` thành các đoạn tăng tối đa. Tổng mỗi đoạn và đáp án là gì?

**Bài 2 (đoán output):** `list(zip([4, 1, 7], [1, 7]))` là gì? `all(x < y for x, y in [(1, 2), (2, 2)])` là gì?

**Bài 3 (điền TODO):** Hoàn thiện lời giải tổng chạy + reset.

In [22]:
def ex3_max_ascending_sum(nums: List[int]) -> int:
    # TODO 1: current = best = nums[0]
    # TODO 2: với k từ 1: nếu nums[k] > nums[k-1] thì cộng, ngược lại reset
    # TODO 3: best = max(best, current)
    pass

# assert ex3_max_ascending_sum([10, 20, 30, 5, 10, 50]) == 65
# assert ex3_max_ascending_sum([3, 3, 3]) == 3

**Bài 4 (tự viết):** Viết `ex4_max_descending_sum(nums)` — tổng lớn nhất của subarray **giảm ngặt**.

In [23]:
def ex4_max_descending_sum(nums: List[int]) -> int:
    # TODO
    pass

# assert ex4_max_descending_sum([10, 20, 30, 5, 10, 50]) == 50
# assert ex4_max_descending_sum([9, 5, 2, 8, 1]) == 16

**Bài 5 (biến thể):** Viết `ex5_best_run(nums)` trả về **chính subarray** tăng ngặt có tổng lớn nhất (nếu hòa, lấy đoạn xuất hiện trước).

In [24]:
def ex5_best_run(nums: List[int]) -> List[int]:
    # TODO: lưu thêm chỉ số bắt đầu start của đoạn hiện tại và của đoạn tốt nhất
    pass

# assert ex5_best_run([10, 20, 30, 5, 10, 50]) == [5, 10, 50]
# assert ex5_best_run([12, 17, 15, 13, 10, 11, 12]) == [10, 11, 12]
# assert ex5_best_run([4, 4]) == [4]

**Bài 6 (mini challenge):** Đoạn "**không giảm**" (cho phép bằng): tìm tổng lớn nhất của subarray không giảm, nhưng **mỗi giá trị chỉ được cộng một lần** nếu lặp liên tiếp (ví dụ `[2, 2, 3]` tính `2 + 3 = 5`).

In [25]:
def ex6_non_decreasing_unique_sum(nums: List[int]) -> int:
    # TODO
    pass

# assert ex6_non_decreasing_unique_sum([2, 2, 3]) == 5
# assert ex6_non_decreasing_unique_sum([1, 2, 2, 3, 1, 9]) == 10
# assert ex6_non_decreasing_unique_sum([5, 5, 5]) == 5

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | cắt khi `nums[k] <= nums[k-1]` | 3 đoạn | `[5,6]`, `[1,2,3]`, `[2,9]` |
| 2 | `zip` dừng ở list ngắn hơn | `2 < 2` sai | `[(4,1),(1,7)]`, `False` |
| 3 | xem mục 5 | khởi tạo bằng `nums[0]` | reset về `nums[k]` |
| 4 | đảo dấu so sánh | `nums[k] < nums[k-1]` | giữ nguyên phần còn lại |
| 5 | lưu `start` | reset thì `start = k` | khi `current > best` lưu `best_l, best_r` |
| 6 | 3 nhánh: `>`, `==`, `<` | `==`: không cộng, không reset | `<`: reset |

## 17. Đáp án tham khảo

**Bài 1:** `[5,6]`=11, `[1,2,3]`=6, `[2,9]`=11 → đáp án **11**.

**Bài 2:** `[(4, 1), (1, 7)]` và `False`.

In [26]:
assert max_ascending_sum_runs([5, 6, 1, 2, 3, 2, 9]) == 11
print(list(zip([4, 1, 7], [1, 7])), all(x < y for x, y in [(1, 2), (2, 2)]))

def ex3_max_ascending_sum_ans(nums):
    current = best = nums[0]
    for k in range(1, len(nums)):
        current = current + nums[k] if nums[k] > nums[k - 1] else nums[k]
        best = max(best, current)
    return best

def ex4_max_descending_sum_ans(nums):
    current = best = nums[0]
    for k in range(1, len(nums)):
        current = current + nums[k] if nums[k] < nums[k - 1] else nums[k]
        best = max(best, current)
    return best

def ex5_best_run_ans(nums):
    current = best = nums[0]
    start = best_l = best_r = 0
    for k in range(1, len(nums)):
        if nums[k] > nums[k - 1]:
            current += nums[k]
        else:
            current = nums[k]
            start = k
        if current > best:
            best, best_l, best_r = current, start, k
    return nums[best_l:best_r + 1]

def ex6_non_decreasing_unique_sum_ans(nums):
    current = best = nums[0]
    for k in range(1, len(nums)):
        if nums[k] > nums[k - 1]:
            current += nums[k]
        elif nums[k] < nums[k - 1]:
            current = nums[k]
        # bằng nhau: giữ nguyên current
        best = max(best, current)
    return best

assert ex3_max_ascending_sum_ans([10, 20, 30, 5, 10, 50]) == 65
assert ex3_max_ascending_sum_ans([3, 3, 3]) == 3
assert ex4_max_descending_sum_ans([10, 20, 30, 5, 10, 50]) == 50
assert ex4_max_descending_sum_ans([9, 5, 2, 8, 1]) == 16
assert ex5_best_run_ans([10, 20, 30, 5, 10, 50]) == [5, 10, 50]
assert ex5_best_run_ans([12, 17, 15, 13, 10, 11, 12]) == [10, 11, 12]
assert ex5_best_run_ans([4, 4]) == [4]
assert ex6_non_decreasing_unique_sum_ans([2, 2, 3]) == 5
assert ex6_non_decreasing_unique_sum_ans([1, 2, 2, 3, 1, 9]) == 10
assert ex6_non_decreasing_unique_sum_ans([5, 5, 5]) == 5
print("Đáp án tham khảo PASS toàn bộ test!")

[(4, 1), (1, 7)] False
Đáp án tham khảo PASS toàn bộ test!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def maxAscendingSum(self, nums: List[int]) -> int:
        current = best = nums[0]
        for k in range(1, len(nums)):
            if nums[k] > nums[k - 1]:     # tăng NGẶT: dấu >
                current += nums[k]        # nối tiếp đoạn
            else:
                current = nums[k]         # gãy: mở đoạn mới từ chính nums[k]
            best = max(best, current)     # cập nhật kỷ lục MỖI bước
        return best
```

**Khung "đoạn liên tiếp một lượt":** `trạng thái đoạn kết thúc tại k` → nối tiếp hoặc reset → cập nhật `best`.

**Checklist:**

- [ ] Dùng `>` (ngặt), không phải `>=`?
- [ ] Reset về `nums[k]`, không phải `0`?
- [ ] Cập nhật `best` sau **mỗi** bước (kể cả đoạn cuối)?
- [ ] Khởi tạo bằng `nums[0]`, vòng lặp từ `k = 1`?
- [ ] Test mảng 1 phần tử, toàn bằng nhau, giảm dần?